In [1]:
# ============================================================
# MultiModalCancerTherapy
# 05 - Multimodal Fusion and Evaluation
# Step 1 - Load Saved Models and Dataset Artifacts
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import torch

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print("Project Root:")
print(PROJECT_ROOT)

# ------------------------------------------------------------
# Clinical model artifacts
# ------------------------------------------------------------

CLINICAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "clinical"
)

clinical_model_path = (
    CLINICAL_MODEL_DIR
    / "clinical_model.joblib"
)

clinical_preprocessor_path = (
    CLINICAL_MODEL_DIR
    / "clinical_preprocessor.joblib"
)

clinical_encoder_path = (
    CLINICAL_MODEL_DIR
    / "target_encoder.joblib"
)

# ------------------------------------------------------------
# Ultrasound model artifacts
# ------------------------------------------------------------

IMAGE_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "image"
)

ultrasound_model_path = (
    IMAGE_MODEL_DIR
    / "ultrasound_unet.pth"
)

ultrasound_info_path = (
    IMAGE_MODEL_DIR
    / "ultrasound_model_info.json"
)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

artifact_paths = {
    "Clinical Model": clinical_model_path,
    "Clinical Preprocessor": clinical_preprocessor_path,
    "Clinical Target Encoder": clinical_encoder_path,
    "Ultrasound Model": ultrasound_model_path,
    "Ultrasound Metadata": ultrasound_info_path
}

print("\nArtifact verification:")
print("-" * 60)

for name, path in artifact_paths.items():
    print(
        f"{name:25} : "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

# ------------------------------------------------------------
# Load clinical artifacts
# ------------------------------------------------------------

clinical_model = joblib.load(
    clinical_model_path
)

clinical_preprocessor = joblib.load(
    clinical_preprocessor_path
)

clinical_target_encoder = joblib.load(
    clinical_encoder_path
)

# ------------------------------------------------------------
# Load ultrasound metadata
# ------------------------------------------------------------

with open(
    ultrasound_info_path,
    "r",
    encoding="utf-8"
) as file:
    ultrasound_metadata = json.load(file)

print("\nClinical model loaded successfully.")

print("\nClinical classes:")
print(
    list(
        clinical_target_encoder.classes_
    )
)

print("\nUltrasound model metadata:")
print(
    json.dumps(
        ultrasound_metadata,
        indent=2
    )
)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:")
print(device)

Project Root:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure

Artifact verification:
------------------------------------------------------------
Clinical Model            : FOUND
Clinical Preprocessor     : FOUND
Clinical Target Encoder   : FOUND
Ultrasound Model          : FOUND
Ultrasound Metadata       : FOUND

Clinical model loaded successfully.

Clinical classes:
['A', 'AK', 'B', 'C', 'C|I', 'D', 'F', 'FK', 'G', 'GI', 'GK', 'H|K', 'I', 'J', 'K', 'KJ', 'L', 'M', 'MI', 'MK', 'N', 'O', 'P', 'Q', 'R', 'S']

Ultrasound model metadata:
{
  "model_name": "UNetSmall",
  "model_type": "Binary Ultrasound Segmentation",
  "input_channels": 3,
  "output_channels": 1,
  "image_size": [
    224,
    224
  ],
  "epochs": 10,
  "learning_rate": 0.001,
  "batch_size": 8,
  "device": "cpu",
  "training_images": 366,
  "validation_images": 44,
  "test_images": 56,
  "total_processed_images": 466,
  "valid_annotation_images": 466,
  "validation_dice": 0.5392648229996363,
  "v

In [2]:
# ============================================================
# Step 20 - Ultrasound Feature Extraction
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

# ------------------------------------------------------------
# Re-create the same U-Net architecture
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetSmall(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up3 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(
            32, 16, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(32, 16)

        self.output = nn.Conv2d(
            16, 1, kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        b = self.bottleneck(p3)

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# ------------------------------------------------------------
# Load trained U-Net
# ------------------------------------------------------------

image_model = UNetSmall(
    in_channels=3,
    out_channels=1
).to(device)

image_model.load_state_dict(
    torch.load(
        ultrasound_model_path,
        map_location=device
    )
)

image_model.eval()

print("Trained U-Net loaded successfully.")


# ------------------------------------------------------------
# Image Dataset
# ------------------------------------------------------------

class UltrasoundFeatureDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["processed_image_path"]
        ).convert("RGB")

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        ).permute(2, 0, 1)

        return image_tensor


# ------------------------------------------------------------
# Load processed image manifest
# ------------------------------------------------------------

processed_manifest_path = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "processed_manifest.csv"
)

fusion_image_df = pd.read_csv(
    processed_manifest_path
)

print("\nProcessed images:", len(fusion_image_df))


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

feature_dataset = UltrasoundFeatureDataset(
    fusion_image_df
)

feature_loader = DataLoader(
    feature_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# Extract bottleneck features
# ------------------------------------------------------------

image_features = []
image_feature_rows = []

with torch.no_grad():

    for start_index, images in enumerate(feature_loader):

        images = images.to(device)

        # Encoder
        e1 = image_model.enc1(images)
        p1 = image_model.pool(e1)

        e2 = image_model.enc2(p1)
        p2 = image_model.pool(e2)

        e3 = image_model.enc3(p2)
        p3 = image_model.pool(e3)

        # Bottleneck representation
        bottleneck = image_model.bottleneck(p3)

        # Global average pooling
        features = torch.mean(
            bottleneck,
            dim=(2, 3)
        )

        image_features.append(
            features.cpu().numpy()
        )


# Combine all batches
image_features = np.concatenate(
    image_features,
    axis=0
)

print("\nImage feature extraction completed.")

print("Feature matrix shape:")
print(image_features.shape)


# ------------------------------------------------------------
# Create feature dataframe
# ------------------------------------------------------------

feature_columns = [
    f"image_feature_{i+1}"
    for i in range(image_features.shape[1])
]

image_feature_df = pd.DataFrame(
    image_features,
    columns=feature_columns
)

image_feature_df.insert(
    0,
    "image_filename",
    fusion_image_df["image_filename"].values
)

image_feature_df.insert(
    0,
    "case_number",
    fusion_image_df["case_number"].values
)

display(
    image_feature_df.head()
)


# ------------------------------------------------------------
# Save image feature representations
# ------------------------------------------------------------

IMAGE_FEATURE_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
)

IMAGE_FEATURE_PATH = (
    IMAGE_FEATURE_DIR
    / "image_features.csv"
)

image_feature_df.to_csv(
    IMAGE_FEATURE_PATH,
    index=False,
    encoding="utf-8"
)

print("\nImage feature file saved:")
print(IMAGE_FEATURE_PATH)

Trained U-Net loaded successfully.

Processed images: 466

Image feature extraction completed.
Feature matrix shape:
(466, 128)


,case_number,image_filename,image_feature_1,image_feature_2,image_feature_3,image_feature_4,image_feature_5,image_feature_6,image_feature_7,image_feature_8,...,image_feature_119,image_feature_120,image_feature_121,image_feature_122,image_feature_123,image_feature_124,image_feature_125,image_feature_126,image_feature_127,image_feature_128
0,1,1_1.jpg,0.371107,0.118345,0.660664,0.399128,0.485627,0.369127,0.310911,0.462197,...,0.277399,0.298957,0.333261,0.455758,0.350223,0.131583,0.629670,0.373175,0.368802,0.611915
1,1,1_2.jpg,0.371107,0.118345,0.660664,0.399128,0.485627,0.369127,0.310911,0.462197,...,0.277399,0.298957,0.333261,0.455758,0.350223,0.131583,0.629670,0.373175,0.368802,0.611915
2,2,2_1.jpg,0.291288,0.279684,0.599715,0.324194,0.384826,0.288028,0.317185,0.370652,...,0.292659,0.259514,0.406121,0.357592,0.339424,0.242890,0.556621,0.287967,0.293850,0.488320
3,3,3_1.jpg,0.179521,0.233694,0.539236,0.250697,0.347347,0.170772,0.253545,0.257635,...,0.227997,0.212779,0.393260,0.248696,0.411222,0.309789,0.505392,0.181990,0.205854,0.466845
4,4,4_1.jpg,0.257797,0.265808,0.546133,0.294403,0.339706,0.253289,0.257410,0.317028,...,0.229476,0.243231,0.428183,0.320438,0.427033,0.388064,0.525616,0.258843,0.259174,0.426172



Image feature file saved:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\image_features.csv


In [3]:
# ============================================================
# Step 21 - Verify Extracted Ultrasound Features
# ============================================================

from PIL import Image
import numpy as np

# ------------------------------------------------------------
# Check first two images
# ------------------------------------------------------------

sample1 = fusion_image_df.iloc[0]
sample2 = fusion_image_df.iloc[1]

img1 = np.asarray(
    Image.open(
        sample1["processed_image_path"]
    ).convert("RGB"),
    dtype=np.float32
)

img2 = np.asarray(
    Image.open(
        sample2["processed_image_path"]
    ).convert("RGB"),
    dtype=np.float32
)

# Pixel difference
pixel_difference = np.mean(
    np.abs(img1 - img2)
)

print("Image 1:")
print(sample1["image_filename"])

print("\nImage 2:")
print(sample2["image_filename"])

print("\nMean pixel difference:")
print(pixel_difference)

# ------------------------------------------------------------
# Compare extracted features
# ------------------------------------------------------------

features1 = image_feature_df.iloc[0][feature_columns].to_numpy(
    dtype=np.float32
)

features2 = image_feature_df.iloc[1][feature_columns].to_numpy(
    dtype=np.float32
)

feature_difference = np.mean(
    np.abs(features1 - features2)
)

print("\nMean feature difference:")
print(feature_difference)

print("\nFeatures exactly identical:")
print(
    np.array_equal(
        features1,
        features2
    )
)

Image 1:
1_1.jpg

Image 2:
1_2.jpg

Mean pixel difference:
0.0

Mean feature difference:
0.0

Features exactly identical:
True


In [4]:
# ============================================================
# Step 21B - Verify Original and Processed Ultrasound Images
# ============================================================

from pathlib import Path
from PIL import Image
import hashlib
import numpy as np

def file_hash(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Select first two images
# ------------------------------------------------------------

row1 = fusion_image_df[
    fusion_image_df["image_filename"] == "1_1.jpg"
].iloc[0]

row2 = fusion_image_df[
    fusion_image_df["image_filename"] == "1_2.jpg"
].iloc[0]


# ------------------------------------------------------------
# Original paths
# ------------------------------------------------------------

original_root = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "ultrasound"
    / "images"
    / "Thyroid Ultrasound"
)

original1 = original_root / "1_1.jpg"
original2 = original_root / "1_2.jpg"

processed1 = Path(
    row1["processed_image_path"]
)

processed2 = Path(
    row2["processed_image_path"]
)


# ------------------------------------------------------------
# Print paths
# ------------------------------------------------------------

print("Original Image 1:")
print(original1)

print("\nOriginal Image 2:")
print(original2)

print("\nProcessed Image 1:")
print(processed1)

print("\nProcessed Image 2:")
print(processed2)


# ------------------------------------------------------------
# File information
# ------------------------------------------------------------

print("\nFile sizes:")
print("Original 1:", original1.stat().st_size, "bytes")
print("Original 2:", original2.stat().st_size, "bytes")

print("\nOriginal MD5:")
print("1_1.jpg:", file_hash(original1))
print("1_2.jpg:", file_hash(original2))

print("\nProcessed MD5:")
print("1_1.jpg:", file_hash(processed1))
print("1_2.jpg:", file_hash(processed2))


# ------------------------------------------------------------
# Compare image pixels
# ------------------------------------------------------------

img1 = np.array(
    Image.open(original1).convert("RGB")
)

img2 = np.array(
    Image.open(original2).convert("RGB")
)

print("\nOriginal image shapes:")
print("1_1:", img1.shape)
print("1_2:", img2.shape)

print("\nOriginal pixel difference:")
print(
    np.mean(
        np.abs(
            img1.astype(np.float32)
            - img2.astype(np.float32)
        )
    )
)

print("\nOriginal images exactly identical:")
print(
    np.array_equal(img1, img2)
)

Original Image 1:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\raw\ultrasound\images\Thyroid Ultrasound\1_1.jpg

Original Image 2:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\raw\ultrasound\images\Thyroid Ultrasound\1_2.jpg

Processed Image 1:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\images\1_1.jpg

Processed Image 2:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\images\1_2.jpg

File sizes:
Original 1: 29865 bytes
Original 2: 29865 bytes

Original MD5:
1_1.jpg: d0403a98817cb5759ecbb0fe7ad671c7
1_2.jpg: d0403a98817cb5759ecbb0fe7ad671c7

Processed MD5:
1_1.jpg: 9ba0ebc3ba179074b6db3df723741059
1_2.jpg: 9ba0ebc3ba179074b6db3df723741059

Original image shapes:
1_1: (360, 560, 3)
1_2: (360, 560, 3)

Original pixel difference:
0.0

Original images exactly identical:
True


In [5]:
# ============================================================
# Step 21C - Detect Duplicate Ultrasound Images
# ============================================================

import hashlib
from collections import defaultdict
from pathlib import Path

def md5_hash(file_path):
    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


# ------------------------------------------------------------
# Calculate image hashes
# ------------------------------------------------------------

image_hash_groups = defaultdict(list)

for _, row in fusion_image_df.iterrows():

    image_path = Path(
        row["processed_image_path"]
    )

    if image_path.exists():

        file_hash_value = md5_hash(
            image_path
        )

        image_hash_groups[
            file_hash_value
        ].append(
            row["image_filename"]
        )


# ------------------------------------------------------------
# Keep only duplicate groups
# ------------------------------------------------------------

duplicate_groups = {
    hash_value: filenames
    for hash_value, filenames
    in image_hash_groups.items()
    if len(filenames) > 1
}


print("Total processed images:")
print(len(fusion_image_df))

print("\nUnique image contents:")
print(len(image_hash_groups))

print("\nDuplicate image groups:")
print(len(duplicate_groups))

print("\nTotal images belonging to duplicate groups:")

duplicate_image_count = sum(
    len(filenames)
    for filenames in duplicate_groups.values()
)

print(duplicate_image_count)


# ------------------------------------------------------------
# Show duplicate groups
# ------------------------------------------------------------

print("\nSample duplicate groups:")
print("=" * 70)

for i, filenames in enumerate(
    list(duplicate_groups.values())[:20],
    start=1
):
    print(f"Group {i}:")
    print(filenames)
    print()

Total processed images:
466

Unique image contents:
459

Duplicate image groups:
4

Total images belonging to duplicate groups:
11

Sample duplicate groups:
Group 1:
['1_1.jpg', '1_2.jpg']

Group 2:
['92_4.jpg', '92_3.jpg']

Group 3:
['150_1.jpg', '233_2.jpg', '323_1.jpg', '326_1.jpg', '349_1.jpg']

Group 4:
['353_2.jpg', '396_1.jpg']



In [7]:
# ============================================================
# Step 21D - Check Duplicate Images and Their Masks
# ============================================================

import hashlib
from pathlib import Path
from collections import defaultdict

def md5_hash(file_path):
    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


# ------------------------------------------------------------
# Build image duplicate groups
# ------------------------------------------------------------

image_groups = defaultdict(list)

for _, row in fusion_image_df.iterrows():

    image_path = Path(
        row["processed_image_path"]
    )

    if image_path.exists():

        image_hash = md5_hash(image_path)

        image_groups[image_hash].append(row)


duplicate_groups = {
    image_hash: rows
    for image_hash, rows in image_groups.items()
    if len(rows) > 1
}


# ------------------------------------------------------------
# Compare masks within each duplicate image group
# ------------------------------------------------------------

print("Duplicate image groups:")
print("=" * 70)

for group_number, (image_hash, rows) in enumerate(
    duplicate_groups.items(),
    start=1
):

    print(f"\nGroup {group_number}")

    mask_hashes = []

    for row in rows:

        mask_path = Path(
            row["processed_mask_path"]
        )

        mask_hash = (
            md5_hash(mask_path)
            if mask_path.exists()
            else "MISSING"
        )

        mask_hashes.append(mask_hash)

        print(
            f"Case: {row['case_number']} | "
            f"Image: {row['image_filename']} | "
            f"Mask: {Path(row['processed_mask_path']).name}"
        )

        print(
            f"  Image hash: {image_hash}"
        )

        print(
            f"  Mask hash : {mask_hash}"
        )

    print(
        "Masks identical:",
        len(set(mask_hashes)) == 1
    )

Duplicate image groups:

Group 1
Case: 1 | Image: 1_1.jpg | Mask: 1_1_mask.png
  Image hash: 9ba0ebc3ba179074b6db3df723741059
  Mask hash : 46ca4a3b5502b3015a4399d1f9b2d9f2
Case: 1 | Image: 1_2.jpg | Mask: 1_2_mask.png
  Image hash: 9ba0ebc3ba179074b6db3df723741059
  Mask hash : 2e6fcb702f8a67839a78c36fea502069
Masks identical: False

Group 2
Case: 92 | Image: 92_4.jpg | Mask: 92_4_mask.png
  Image hash: 1239a8e7222240f3bf897380ac0bcd62
  Mask hash : 467303b547c9f61134e3721346eb9adb
Case: 92 | Image: 92_3.jpg | Mask: 92_3_mask.png
  Image hash: 1239a8e7222240f3bf897380ac0bcd62
  Mask hash : 315ea7cff24f342abe65a64cfe222b22
Masks identical: False

Group 3
Case: 150 | Image: 150_1.jpg | Mask: 150_1_mask.png
  Image hash: 792380e62d4da56c44e8a0988ada4558
  Mask hash : 3e1c7b63a6c76721cc257b7b9574c1ac
Case: 233 | Image: 233_2.jpg | Mask: 233_2_mask.png
  Image hash: 792380e62d4da56c44e8a0988ada4558
  Mask hash : 183225e0e6e439155cf69b3daad12dc5
Case: 323 | Image: 323_1.jpg | Mask: 323_1_ma

In [8]:
# ============================================================
# Step 21E - Create Duplicate-Free Ultrasound Manifest
# ============================================================

from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd


def md5_hash(file_path):
    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


# ------------------------------------------------------------
# Calculate hash for every processed image
# ------------------------------------------------------------

image_hashes = []

for _, row in fusion_image_df.iterrows():

    image_path = Path(
        row["processed_image_path"]
    )

    if image_path.exists():
        image_hashes.append(
            md5_hash(image_path)
        )
    else:
        image_hashes.append(None)


fusion_image_df["image_hash"] = image_hashes


# ------------------------------------------------------------
# Count image content frequency
# ------------------------------------------------------------

hash_counts = (
    fusion_image_df["image_hash"]
    .value_counts()
)


# ------------------------------------------------------------
# Keep only images whose content appears once
# ------------------------------------------------------------

clean_image_df = fusion_image_df[
    fusion_image_df["image_hash"].map(hash_counts) == 1
].copy()


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

duplicate_images_removed = (
    len(fusion_image_df)
    - len(clean_image_df)
)

print("Original processed images:", len(fusion_image_df))

print(
    "Duplicate images removed:",
    duplicate_images_removed
)

print(
    "Clean unique images:",
    len(clean_image_df)
)

print(
    "Remaining duplicate contents:",
    clean_image_df["image_hash"].duplicated().sum()
)


# ------------------------------------------------------------
# Remove temporary hash column
# ------------------------------------------------------------

clean_image_df = clean_image_df.drop(
    columns=["image_hash"]
)


# ------------------------------------------------------------
# Save clean manifest
# ------------------------------------------------------------

CLEAN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_manifest.csv"
)

clean_image_df.to_csv(
    CLEAN_MANIFEST_PATH,
    index=False,
    encoding="utf-8"
)


print("\nClean manifest saved:")
print(CLEAN_MANIFEST_PATH)

print("\nSample:")
display(
    clean_image_df.head(10)
)

Original processed images: 466
Duplicate images removed: 11
Clean unique images: 455
Remaining duplicate contents: 0

Clean manifest saved:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\clean_manifest.csv

Sample:


,case_number,image_number,image_filename,processed_image_path,processed_mask_path,width,height,num_points
2,2,1,2_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,50
3,3,1,3_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,30
4,4,1,4_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,75
5,5,1,5_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,55
6,6,1,6_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,33
7,7,1,7_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,50
8,8,1,8_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,49
9,9,1,9_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,31
10,10,1,10_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,58
11,11,1,11_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,224,224,47


In [10]:
# ============================================================
# Step 2 - Load Clinical Data and Ultrasound Features
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

# ------------------------------------------------------------
# Clinical dataset
# ------------------------------------------------------------

CLINICAL_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "clinical"
    / "Thyroid Disease Data"
    / "thyroidDF.csv"
)

clinical_df = pd.read_csv(
    CLINICAL_DATA_PATH
)

# ------------------------------------------------------------
# Ultrasound feature file
# ------------------------------------------------------------

IMAGE_FEATURE_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "image_features.csv"
)

image_feature_df = pd.read_csv(
    IMAGE_FEATURE_PATH
)

# ------------------------------------------------------------
# Display information
# ------------------------------------------------------------

print("Clinical dataset loaded:")
print("Rows   :", len(clinical_df))
print("Columns:", len(clinical_df.columns))

print("\nUltrasound feature dataset loaded:")
print("Rows   :", len(image_feature_df))
print("Columns:", len(image_feature_df.columns))

print("\nUltrasound feature columns:")
print(
    image_feature_df.columns[:10].tolist()
)

print(
    "...",
    image_feature_df.columns[-5:].tolist()
)

print("\nClinical columns:")
print(
    clinical_df.columns.tolist()
)

print("\nFirst ultrasound feature records:")
display(
    image_feature_df.head(5)
)

Clinical dataset loaded:
Rows   : 9172
Columns: 31

Ultrasound feature dataset loaded:
Rows   : 466
Columns: 130

Ultrasound feature columns:
['case_number', 'image_filename', 'image_feature_1', 'image_feature_2', 'image_feature_3', 'image_feature_4', 'image_feature_5', 'image_feature_6', 'image_feature_7', 'image_feature_8']
... ['image_feature_124', 'image_feature_125', 'image_feature_126', 'image_feature_127', 'image_feature_128']

Clinical columns:
['age', 'sex', 'on_thyroxine', 'query_on_thyroxine', 'on_antithyroid_meds', 'sick', 'pregnant', 'thyroid_surgery', 'I131_treatment', 'query_hypothyroid', 'query_hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH_measured', 'TSH', 'T3_measured', 'T3', 'TT4_measured', 'TT4', 'T4U_measured', 'T4U', 'FTI_measured', 'FTI', 'TBG_measured', 'TBG', 'referral_source', 'target', 'patient_id']

First ultrasound feature records:


,case_number,image_filename,image_feature_1,image_feature_2,image_feature_3,image_feature_4,image_feature_5,image_feature_6,image_feature_7,image_feature_8,...,image_feature_119,image_feature_120,image_feature_121,image_feature_122,image_feature_123,image_feature_124,image_feature_125,image_feature_126,image_feature_127,image_feature_128
0,1,1_1.jpg,0.371107,0.118345,0.660664,0.399128,0.485627,0.369127,0.310911,0.462197,...,0.277399,0.298957,0.333261,0.455758,0.350223,0.131583,0.629670,0.373175,0.368802,0.611915
1,1,1_2.jpg,0.371107,0.118345,0.660664,0.399128,0.485627,0.369127,0.310911,0.462197,...,0.277399,0.298957,0.333261,0.455758,0.350223,0.131583,0.629670,0.373175,0.368802,0.611915
2,2,2_1.jpg,0.291288,0.279684,0.599715,0.324194,0.384826,0.288028,0.317185,0.370652,...,0.292659,0.259514,0.406121,0.357592,0.339424,0.242890,0.556621,0.287967,0.293850,0.488320
3,3,3_1.jpg,0.179521,0.233694,0.539236,0.250697,0.347347,0.170772,0.253545,0.257635,...,0.227997,0.212779,0.393260,0.248696,0.411222,0.309789,0.505392,0.181990,0.205854,0.466845
4,4,4_1.jpg,0.257797,0.265808,0.546133,0.294403,0.339706,0.253289,0.257410,0.317028,...,0.229476,0.243231,0.428183,0.320438,0.427033,0.388064,0.525616,0.258843,0.259174,0.426172


In [11]:
# ============================================================
# Step 3 - Extract Features Using Final Clean U-Net
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from PIL import Image


# ------------------------------------------------------------
# Load clean manifest
# ------------------------------------------------------------

CLEAN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_manifest.csv"
)

clean_feature_df = pd.read_csv(
    CLEAN_MANIFEST_PATH
)

print("Clean ultrasound images:", len(clean_feature_df))


# ------------------------------------------------------------
# Final clean U-Net architecture
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetSmall(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up3 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(
            32, 16, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(32, 16)

        self.output = nn.Conv2d(
            16, 1, kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        b = self.bottleneck(p3)

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# ------------------------------------------------------------
# Load final clean model
# ------------------------------------------------------------

FINAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "ultrasound_unet_clean.pth"
)

image_model = UNetSmall(
    in_channels=3,
    out_channels=1
).to(device)

image_model.load_state_dict(
    torch.load(
        FINAL_MODEL_PATH,
        map_location=device
    )
)

image_model.eval()

print("Final clean U-Net loaded successfully.")


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class CleanImageDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["processed_image_path"]
        ).convert("RGB")

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        ).permute(2, 0, 1)

        return image_tensor


clean_dataset = CleanImageDataset(
    clean_feature_df
)

clean_loader = DataLoader(
    clean_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# Extract 128-dimensional image representation
# ------------------------------------------------------------

all_image_features = []

with torch.no_grad():

    for images in clean_loader:

        images = images.to(device)

        # Encoder
        e1 = image_model.enc1(images)
        p1 = image_model.pool(e1)

        e2 = image_model.enc2(p1)
        p2 = image_model.pool(e2)

        e3 = image_model.enc3(p2)
        p3 = image_model.pool(e3)

        # Bottleneck
        bottleneck = image_model.bottleneck(p3)

        # Global Average Pooling
        features = torch.mean(
            bottleneck,
            dim=(2, 3)
        )

        all_image_features.append(
            features.cpu().numpy()
        )


all_image_features = np.concatenate(
    all_image_features,
    axis=0
)

print("\nFinal image feature extraction completed.")
print(
    "Feature matrix shape:",
    all_image_features.shape
)


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

image_feature_columns = [
    f"image_feature_{i+1}"
    for i in range(
        all_image_features.shape[1]
    )
]

clean_image_features = pd.DataFrame(
    all_image_features,
    columns=image_feature_columns
)

clean_image_features.insert(
    0,
    "image_filename",
    clean_feature_df["image_filename"].values
)

clean_image_features.insert(
    0,
    "case_number",
    clean_feature_df["case_number"].values
)


# ------------------------------------------------------------
# Save final clean image features
# ------------------------------------------------------------

FINAL_IMAGE_FEATURE_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_image_features.csv"
)

clean_image_features.to_csv(
    FINAL_IMAGE_FEATURE_PATH,
    index=False,
    encoding="utf-8"
)

print("\nFinal clean image features saved:")
print(FINAL_IMAGE_FEATURE_PATH)

print("\nSample:")
display(
    clean_image_features.head()
)

Clean ultrasound images: 455
Final clean U-Net loaded successfully.

Final image feature extraction completed.
Feature matrix shape: (455, 128)

Final clean image features saved:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\clean_image_features.csv

Sample:


,case_number,image_filename,image_feature_1,image_feature_2,image_feature_3,image_feature_4,image_feature_5,image_feature_6,image_feature_7,image_feature_8,...,image_feature_119,image_feature_120,image_feature_121,image_feature_122,image_feature_123,image_feature_124,image_feature_125,image_feature_126,image_feature_127,image_feature_128
0,2,2_1.jpg,0.342251,0.488708,0.480971,0.285028,0.471456,0.395482,0.407041,0.350835,...,0.579056,0.496462,0.483437,0.357522,0.277624,0.416651,0.423303,0.424878,0.346846,0.421076
1,3,3_1.jpg,0.301097,0.505569,0.534111,0.393278,0.481612,0.307898,0.191481,0.267013,...,0.611042,0.514620,0.537195,0.418712,0.362350,0.177747,0.309070,0.452530,0.459297,0.187657
2,4,4_1.jpg,0.302944,0.508795,0.528774,0.403691,0.459762,0.379977,0.340536,0.298445,...,0.500117,0.501477,0.487248,0.402929,0.283510,0.332253,0.358887,0.424076,0.369682,0.324829
3,5,5_1.jpg,0.259129,0.590523,0.560816,0.364924,0.506134,0.377654,0.411568,0.298040,...,0.615895,0.569465,0.579169,0.417497,0.342363,0.408067,0.355404,0.382435,0.428180,0.394156
4,6,6_1.jpg,0.261385,0.562208,0.527091,0.348736,0.457514,0.349532,0.266268,0.258792,...,0.565441,0.521939,0.517564,0.408761,0.347682,0.282799,0.309477,0.380533,0.419178,0.272659


In [12]:
# ============================================================
# Step 4 - Check Clinical ↔ Ultrasound Pairing
# ============================================================

# Ultrasound case numbers
ultrasound_cases = set(
    clean_image_features["case_number"]
)

print("Ultrasound unique cases:")
print(len(ultrasound_cases))

print("\nSample ultrasound case numbers:")
print(sorted(ultrasound_cases)[:20])


# Clinical patient IDs
clinical_patient_ids = set(
    clinical_df["patient_id"]
)

print("\nClinical unique patient IDs:")
print(len(clinical_patient_ids))

print("\nSample clinical patient IDs:")
print(sorted(clinical_patient_ids)[:20])


# ------------------------------------------------------------
# Direct numeric intersection check
# ------------------------------------------------------------

numeric_intersection = (
    ultrasound_cases
    & clinical_patient_ids
)

print("\nDirect ID intersection:")
print(len(numeric_intersection))

print("\nCommon IDs:")
print(
    sorted(numeric_intersection)[:20]
)

Ultrasound unique cases:
374

Sample ultrasound case numbers:
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

Clinical unique patient IDs:
9172

Sample clinical patient IDs:
[840801013, 840801014, 840801042, 840803046, 840803047, 840803048, 840803068, 840807019, 840808060, 840808073, 840810016, 840813022, 840813060, 840813068, 840814014, 840814057, 840815016, 840815020, 840815067, 840815068]

Direct ID intersection:
0

Common IDs:
[]


In [13]:
# ============================================================
# Step 5 - Ultrasound TIRADS Label Analysis
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load original XML-image mapping
# ------------------------------------------------------------

MAPPING_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "mapping.csv"
)

mapping_df = pd.read_csv(
    MAPPING_PATH
)

print("Total mapping records:", len(mapping_df))

# ------------------------------------------------------------
# Load clean unique image manifest
# ------------------------------------------------------------

CLEAN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_manifest.csv"
)

clean_df = pd.read_csv(
    CLEAN_MANIFEST_PATH
)

print("Clean ultrasound images:", len(clean_df))

# ------------------------------------------------------------
# Keep only clean images
# Match using image filename
# ------------------------------------------------------------

tirads_df = clean_df[
    [
        "case_number",
        "image_filename",
        "processed_image_path",
        "processed_mask_path"
    ]
].merge(
    mapping_df[
        [
            "case_number",
            "image_filename",
            "tirads"
        ]
    ],
    on=[
        "case_number",
        "image_filename"
    ],
    how="left"
)

print("\nClean images with TIRADS information:")
print(len(tirads_df))

# ------------------------------------------------------------
# Missing TIRADS
# ------------------------------------------------------------

missing_tirads = tirads_df["tirads"].isna().sum()

print("\nMissing TIRADS labels:")
print(missing_tirads)

print(
    "TIRADS available percentage:",
    round(
        (
            (len(tirads_df) - missing_tirads)
            / len(tirads_df)
        ) * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# TIRADS distribution
# ------------------------------------------------------------

print("\nTIRADS Distribution:")
print("=" * 50)

tirads_counts = (
    tirads_df["tirads"]
    .fillna("MISSING")
    .astype(str)
    .value_counts()
)

print(tirads_counts)

# ------------------------------------------------------------
# Show sample
# ------------------------------------------------------------

print("\nSample records:")
display(
    tirads_df.head(10)
)

Total mapping records: 480
Clean ultrasound images: 455

Clean images with TIRADS information:
455

Missing TIRADS labels:
116
TIRADS available percentage: 74.51 %

TIRADS Distribution:
tirads
MISSING    116
4a          92
4b          76
4c          66
5           45
2           41
3           19
Name: count, dtype: int64

Sample records:


,case_number,image_filename,processed_image_path,processed_mask_path,tirads
0,2,2_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,2
1,3,3_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4a
2,4,4_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4a
3,5,5_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,5
4,6,6_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4b
5,7,7_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4a
6,8,8_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4c
7,9,9_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4b
8,10,10_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,4b
9,11,11_1.jpg,D:\MultiModalCancerTherapy_Structure\MultiModa...,D:\MultiModalCancerTherapy_Structure\MultiModa...,5


In [14]:
# ============================================================
# Step 6 - Prepare TIRADS Classification Dataset
#         and Case-Level Train/Validation/Test Split
# ============================================================

from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Keep only images with valid TIRADS labels
# ------------------------------------------------------------

tirads_labeled_df = tirads_df[
    tirads_df["tirads"].notna()
].copy()

tirads_labeled_df["tirads"] = (
    tirads_labeled_df["tirads"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep expected TIRADS categories only
valid_tirads_classes = [
    "2",
    "3",
    "4a",
    "4b",
    "4c",
    "5"
]

tirads_labeled_df = tirads_labeled_df[
    tirads_labeled_df["tirads"].isin(
        valid_tirads_classes
    )
].copy()

print("TIRADS labeled images:")
print(len(tirads_labeled_df))

print("\nTIRADS classes:")
print(
    sorted(
        tirads_labeled_df["tirads"].unique()
    )
)

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print("\nTIRADS class distribution:")
print(
    tirads_labeled_df["tirads"].value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Case-level split
# ------------------------------------------------------------

gss_train = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    gss_train.split(
        tirads_labeled_df,
        groups=tirads_labeled_df["case_number"]
    )
)

tirads_train_df = tirads_labeled_df.iloc[
    train_idx
].copy()

tirads_temp_df = tirads_labeled_df.iloc[
    temp_idx
].copy()

# 50% of temporary -> validation
# 50% of temporary -> test

gss_val_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_val_test.split(
        tirads_temp_df,
        groups=tirads_temp_df["case_number"]
    )
)

tirads_val_df = tirads_temp_df.iloc[
    val_idx
].copy()

tirads_test_df = tirads_temp_df.iloc[
    test_idx
].copy()

# ------------------------------------------------------------
# Verify no case overlap
# ------------------------------------------------------------

train_cases = set(
    tirads_train_df["case_number"]
)

val_cases = set(
    tirads_val_df["case_number"]
)

test_cases = set(
    tirads_test_df["case_number"]
)

print("\nTIRADS dataset split:")
print("-" * 60)

print(
    "Training images   :",
    len(tirads_train_df)
)

print(
    "Validation images :",
    len(tirads_val_df)
)

print(
    "Testing images    :",
    len(tirads_test_df)
)

print("\nUnique cases:")
print(
    "Training cases   :",
    len(train_cases)
)

print(
    "Validation cases :",
    len(val_cases)
)

print(
    "Testing cases    :",
    len(test_cases)
)

print("\nCase overlap:")
print(
    "Train ∩ Validation:",
    len(train_cases & val_cases)
)

print(
    "Train ∩ Test      :",
    len(train_cases & test_cases)
)

print(
    "Validation ∩ Test :",
    len(val_cases & test_cases)
)

# ------------------------------------------------------------
# Show class distribution in each split
# ------------------------------------------------------------

print("\nTraining class distribution:")
print(
    tirads_train_df["tirads"]
    .value_counts()
    .sort_index()
)

print("\nValidation class distribution:")
print(
    tirads_val_df["tirads"]
    .value_counts()
    .sort_index()
)

print("\nTesting class distribution:")
print(
    tirads_test_df["tirads"]
    .value_counts()
    .sort_index()
)

TIRADS labeled images:
339

TIRADS classes:
['2', '3', '4a', '4b', '4c', '5']

TIRADS class distribution:
tirads
2     41
3     19
4a    92
4b    76
4c    66
5     45
Name: count, dtype: int64

TIRADS dataset split:
------------------------------------------------------------
Training images   : 271
Validation images : 31
Testing images    : 37

Unique cases:
Training cases   : 232
Validation cases : 29
Testing cases    : 30

Case overlap:
Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0

Training class distribution:
tirads
2     30
3     13
4a    73
4b    60
4c    56
5     39
Name: count, dtype: int64

Validation class distribution:
tirads
2     7
4a    5
4b    9
4c    7
5     3
Name: count, dtype: int64

Testing class distribution:
tirads
2      4
3      6
4a    14
4b     7
4c     3
5      3
Name: count, dtype: int64


In [15]:
# ============================================================
# Step 7 - TIRADS Image Dataset and DataLoaders
# ============================================================

import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from PIL import Image


# ------------------------------------------------------------
# TIRADS label encoding
# ------------------------------------------------------------

tirads_classes = [
    "2",
    "3",
    "4a",
    "4b",
    "4c",
    "5"
]

tirads_to_index = {
    label: index
    for index, label in enumerate(tirads_classes)
}

index_to_tirads = {
    index: label
    for label, index in tirads_to_index.items()
}

print("TIRADS encoding:")
print(tirads_to_index)


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class TIRADSImageDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Load image
        image = Image.open(
            row["processed_image_path"]
        ).convert("RGB")

        # Convert to numpy and normalize
        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        # HWC -> CHW
        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        ).permute(2, 0, 1)

        # TIRADS label
        label = tirads_to_index[
            str(row["tirads"])
        ]

        label_tensor = torch.tensor(
            label,
            dtype=torch.long
        )

        return image_tensor, label_tensor


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

tirads_train_dataset = TIRADSImageDataset(
    tirads_train_df
)

tirads_val_dataset = TIRADSImageDataset(
    tirads_val_df
)

tirads_test_dataset = TIRADSImageDataset(
    tirads_test_df
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

TIRADS_BATCH_SIZE = 8

tirads_train_loader = DataLoader(
    tirads_train_dataset,
    batch_size=TIRADS_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

tirads_val_loader = DataLoader(
    tirads_val_dataset,
    batch_size=TIRADS_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

tirads_test_loader = DataLoader(
    tirads_test_dataset,
    batch_size=TIRADS_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# Verify one batch
# ------------------------------------------------------------

sample_images, sample_labels = next(
    iter(tirads_train_loader)
)

print("\nTIRADS DataLoader created successfully.")

print("\nTraining samples:")
print(len(tirads_train_dataset))

print("Validation samples:")
print(len(tirads_val_dataset))

print("Testing samples:")
print(len(tirads_test_dataset))

print("\nImage batch shape:")
print(sample_images.shape)

print("\nLabel batch shape:")
print(sample_labels.shape)

print("\nUnique labels in sample batch:")
print(torch.unique(sample_labels))

print("\nSample TIRADS labels:")
print([
    index_to_tirads[int(label)]
    for label in sample_labels[:8]
])

TIRADS encoding:
{'2': 0, '3': 1, '4a': 2, '4b': 3, '4c': 4, '5': 5}

TIRADS DataLoader created successfully.

Training samples:
271
Validation samples:
31
Testing samples:
37

Image batch shape:
torch.Size([8, 3, 224, 224])

Label batch shape:
torch.Size([8])

Unique labels in sample batch:
tensor([0, 2, 3, 4])

Sample TIRADS labels:
['4a', '4c', '4c', '2', '4a', '4a', '4b', '4c']


In [16]:
# ============================================================
# Step 8 - TIRADS Classification Model
# ============================================================

import torch
import torch.nn as nn
import numpy as np

# ------------------------------------------------------------
# TIRADS class count
# ------------------------------------------------------------

NUM_TIRADS_CLASSES = len(tirads_classes)

print("Number of TIRADS classes:", NUM_TIRADS_CLASSES)
print("Classes:", tirads_classes)


# ------------------------------------------------------------
# TIRADS Classifier using U-Net encoder
# ------------------------------------------------------------

class TIRADSClassifier(nn.Module):

    def __init__(self, num_classes=6):
        super().__init__()

        # Same encoder architecture used in U-Net
        self.enc1 = DoubleConv(3, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        # Deep image representation
        self.bottleneck = DoubleConv(64, 128)

        # Classification head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Dropout(0.30),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(inplace=True),

            nn.Dropout(0.20),

            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        # Bottleneck
        features = self.bottleneck(p3)

        # Global average pooling
        pooled = self.global_pool(features)

        # Classification
        output = self.classifier(pooled)

        return output


# ------------------------------------------------------------
# Create classifier
# ------------------------------------------------------------

tirads_model = TIRADSClassifier(
    num_classes=NUM_TIRADS_CLASSES
).to(device)


# ------------------------------------------------------------
# Load trained U-Net encoder weights
# ------------------------------------------------------------

FINAL_UNET_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "ultrasound_unet_clean.pth"
)

# Load the trained U-Net
unet_checkpoint = torch.load(
    FINAL_UNET_PATH,
    map_location=device
)

# Copy compatible encoder weights
encoder_state = {}

for key, value in unet_checkpoint.items():

    if (
        key.startswith("enc1.")
        or key.startswith("enc2.")
        or key.startswith("enc3.")
        or key.startswith("bottleneck.")
    ):
        encoder_state[key] = value


missing_keys, unexpected_keys = (
    tirads_model.load_state_dict(
        encoder_state,
        strict=False
    )
)

print("\nPretrained U-Net encoder weights loaded.")

print("Missing keys:", len(missing_keys))
print("Unexpected keys:", len(unexpected_keys))


# ------------------------------------------------------------
# Class distribution and weights
# ------------------------------------------------------------

train_counts = (
    tirads_train_df["tirads"]
    .value_counts()
)

train_counts = train_counts.reindex(
    tirads_classes
).fillna(0)

print("\nTraining class counts:")
print(train_counts)


# Balanced class weights
total_samples = len(tirads_train_df)

class_weights = (
    total_samples
    / (
        NUM_TIRADS_CLASSES
        * train_counts.values
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:")
print(class_weights)


# ------------------------------------------------------------
# Loss and optimizer
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    tirads_model.parameters(),
    lr=1e-3
)


# ------------------------------------------------------------
# Model statistics
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in tirads_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in tirads_model.parameters()
    if p.requires_grad
)


print("\nTIRADS classifier created successfully.")

print("Device:", device)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ------------------------------------------------------------
# Test forward pass
# ------------------------------------------------------------

sample_images, sample_labels = next(
    iter(tirads_train_loader)
)

sample_images = sample_images.to(device)

with torch.no_grad():

    sample_output = tirads_model(
        sample_images
    )

print("\nInput shape:")
print(sample_images.shape)

print("\nClassifier output shape:")
print(sample_output.shape)

print("\nExpected output shape:")
print(
    (
        sample_images.shape[0],
        NUM_TIRADS_CLASSES
    )
)

Number of TIRADS classes: 6
Classes: ['2', '3', '4a', '4b', '4c', '5']

Pretrained U-Net encoder weights loaded.
Missing keys: 4
Unexpected keys: 0

Training class counts:
tirads
2     30
3     13
4a    73
4b    60
4c    56
5     39
Name: count, dtype: int64

Class weights:
tensor([1.5056, 3.4744, 0.6187, 0.7528, 0.8065, 1.1581])

TIRADS classifier created successfully.
Device: cpu
Total parameters: 303,126
Trainable parameters: 303,126

Input shape:
torch.Size([8, 3, 224, 224])

Classifier output shape:
torch.Size([8, 6])

Expected output shape:
(8, 6)


In [17]:
# ============================================================
# Step 9 - Train TIRADS Classifier
# ============================================================

import time
import pandas as pd
import torch

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

TIRADS_EPOCHS = 10
TIRADS_LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    tirads_model.parameters(),
    lr=TIRADS_LEARNING_RATE
)

print("TIRADS training configuration")
print("=" * 55)
print("Epochs:", TIRADS_EPOCHS)
print("Learning rate:", TIRADS_LEARNING_RATE)
print("Device:", device)


# ------------------------------------------------------------
# Validation function
# ------------------------------------------------------------

def validate_tirads_model(model, loader):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    batches = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            total_loss += loss.item()
            batches += 1

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += labels.size(0)

    avg_loss = total_loss / max(batches, 1)

    accuracy = (
        total_correct / total_samples
        if total_samples > 0
        else 0.0
    )

    return avg_loss, accuracy


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

tirads_history = []

best_val_loss = float("inf")
best_epoch = 0

BEST_TIRADS_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "best_tirads_classifier.pth"
)

print("\nStarting TIRADS training...\n")

for epoch in range(TIRADS_EPOCHS):

    start_time = time.time()

    tirads_model.train()

    running_loss = 0.0
    total_correct = 0
    total_samples = 0
    batches = 0

    for images, labels in tirads_train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = tirads_model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        batches += 1

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

    train_loss = (
        running_loss / max(batches, 1)
    )

    train_accuracy = (
        total_correct / total_samples
        if total_samples > 0
        else 0.0
    )

    # Validation
    val_loss, val_accuracy = validate_tirads_model(
        tirads_model,
        tirads_val_loader
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            tirads_model.state_dict(),
            BEST_TIRADS_PATH
        )

        best_marker = " <-- BEST"

    else:
        best_marker = ""

    tirads_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print(
        f"Epoch [{epoch + 1}/{TIRADS_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {elapsed:.1f}s"
        f"{best_marker}"
    )


print("\nTIRADS training completed successfully.")

print("\nBest epoch:", best_epoch)
print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)

print("\nBest model saved:")
print(BEST_TIRADS_PATH)

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

tirads_history_df = pd.DataFrame(
    tirads_history
)

display(tirads_history_df)

TIRADS training configuration
Epochs: 10
Learning rate: 0.001
Device: cpu

Starting TIRADS training...

Epoch [1/10] Train Loss: 1.7870 | Train Acc: 0.1808 | Val Loss: 1.7773 | Val Acc: 0.1290 | Time: 45.6s <-- BEST
Epoch [2/10] Train Loss: 1.7825 | Train Acc: 0.2030 | Val Loss: 1.7424 | Val Acc: 0.3548 | Time: 44.6s <-- BEST
Epoch [3/10] Train Loss: 1.7726 | Train Acc: 0.2288 | Val Loss: 1.6851 | Val Acc: 0.2258 | Time: 31.4s <-- BEST
Epoch [4/10] Train Loss: 1.7367 | Train Acc: 0.1919 | Val Loss: 1.8084 | Val Acc: 0.1613 | Time: 9.2s
Epoch [5/10] Train Loss: 1.7325 | Train Acc: 0.2399 | Val Loss: 1.7676 | Val Acc: 0.2581 | Time: 9.3s
Epoch [6/10] Train Loss: 1.7205 | Train Acc: 0.2288 | Val Loss: 2.0552 | Val Acc: 0.1613 | Time: 9.9s
Epoch [7/10] Train Loss: 1.7578 | Train Acc: 0.2583 | Val Loss: 1.8876 | Val Acc: 0.0000 | Time: 9.4s
Epoch [8/10] Train Loss: 1.7220 | Train Acc: 0.2546 | Val Loss: 1.8864 | Val Acc: 0.2258 | Time: 9.4s
Epoch [9/10] Train Loss: 1.6789 | Train Acc: 0.291

,epoch,train_loss,train_accuracy,val_loss,val_accuracy
0,1,1.787028,0.180812,1.777292,0.129032
1,2,1.782462,0.202952,1.742386,0.354839
2,3,1.772577,0.228782,1.685119,0.225806
3,4,1.736701,0.191882,1.808372,0.161290
4,5,1.732505,0.239852,1.767622,0.258065
5,6,1.720509,0.228782,2.055187,0.161290
6,7,1.757835,0.258303,1.887593,0.000000
7,8,1.721955,0.254613,1.886405,0.225806
8,9,1.678861,0.291513,2.521573,0.161290
9,10,1.720808,0.269373,1.829270,0.322581


In [18]:
# ============================================================
# Step 10 - Evaluate Best TIRADS Classifier
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np
import torch

# ------------------------------------------------------------
# Load best TIRADS model
# ------------------------------------------------------------

BEST_TIRADS_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "best_tirads_classifier.pth"
)

if not BEST_TIRADS_PATH.exists():
    raise FileNotFoundError(
        f"Best TIRADS model not found:\n{BEST_TIRADS_PATH}"
    )

tirads_model.load_state_dict(
    torch.load(
        BEST_TIRADS_PATH,
        map_location=device
    )
)

tirads_model.eval()

print("Best TIRADS classifier loaded successfully.")
print("Best epoch:", best_epoch)
print("Best validation loss:", f"{best_val_loss:.4f}")


# ------------------------------------------------------------
# Test predictions
# ------------------------------------------------------------

all_true_labels = []
all_pred_labels = []

with torch.no_grad():

    for images, labels in tirads_test_loader:

        images = images.to(device)

        outputs = tirads_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_true_labels.extend(
            labels.numpy()
        )

        all_pred_labels.extend(
            predictions.cpu().numpy()
        )


y_tirads_true = np.array(
    all_true_labels
)

y_tirads_pred = np.array(
    all_pred_labels
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

tirads_accuracy = accuracy_score(
    y_tirads_true,
    y_tirads_pred
)

tirads_precision = precision_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

tirads_recall = recall_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

tirads_f1 = f1_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nTIRADS Test Evaluation")
print("=" * 60)

print(f"Accuracy : {tirads_accuracy:.4f}")
print(f"Precision: {tirads_precision:.4f}")
print(f"Recall   : {tirads_recall:.4f}")
print(f"F1-Score : {tirads_f1:.4f}")


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_tirads_true,
        y_tirads_pred,
        labels=np.arange(len(tirads_classes)),
        target_names=tirads_classes,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_tirads = confusion_matrix(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes))
)

print("\nConfusion Matrix:")
print(cm_tirads)

Best TIRADS classifier loaded successfully.
Best epoch: 3
Best validation loss: 1.6851

TIRADS Test Evaluation
Accuracy : 0.1081
Precision: 0.0197
Recall   : 0.1081
F1-Score : 0.0332

Classification Report:
              precision    recall  f1-score   support

           2       0.12      0.75      0.21         4
           3       0.00      0.00      0.00         6
          4a       0.00      0.00      0.00        14
          4b       0.00      0.00      0.00         7
          4c       0.08      0.33      0.13         3
           5       0.00      0.00      0.00         3

    accuracy                           0.11        37
   macro avg       0.03      0.18      0.06        37
weighted avg       0.02      0.11      0.03        37


Confusion Matrix:
[[3 0 0 0 1 0]
 [6 0 0 0 0 0]
 [7 0 0 0 7 0]
 [4 0 0 0 3 0]
 [2 0 0 0 1 0]
 [3 0 0 0 0 0]]


In [19]:
# ============================================================
# Step 10 - Evaluate Best TIRADS Classifier
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import torch


# ------------------------------------------------------------
# Load best TIRADS model
# ------------------------------------------------------------

BEST_TIRADS_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "best_tirads_classifier.pth"
)

if not BEST_TIRADS_PATH.exists():
    raise FileNotFoundError(
        f"Best TIRADS model not found:\n{BEST_TIRADS_PATH}"
    )

tirads_model.load_state_dict(
    torch.load(
        BEST_TIRADS_PATH,
        map_location=device
    )
)

tirads_model.eval()

print("Best TIRADS classifier loaded successfully.")
print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)


# ------------------------------------------------------------
# Generate test predictions
# ------------------------------------------------------------

all_true_labels = []
all_pred_labels = []

with torch.no_grad():

    for images, labels in tirads_test_loader:

        images = images.to(device)

        outputs = tirads_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_true_labels.extend(
            labels.numpy()
        )

        all_pred_labels.extend(
            predictions.cpu().numpy()
        )


y_tirads_true = np.array(
    all_true_labels
)

y_tirads_pred = np.array(
    all_pred_labels
)


# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

tirads_accuracy = accuracy_score(
    y_tirads_true,
    y_tirads_pred
)

tirads_precision = precision_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

tirads_recall = recall_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

tirads_f1 = f1_score(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# Overall results
# ------------------------------------------------------------

print("\nTIRADS Test Evaluation")
print("=" * 60)

print(
    f"Accuracy : {tirads_accuracy:.4f}"
)

print(
    f"Precision: {tirads_precision:.4f}"
)

print(
    f"Recall   : {tirads_recall:.4f}"
)

print(
    f"F1-Score : {tirads_f1:.4f}"
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_tirads_true,
        y_tirads_pred,
        labels=np.arange(len(tirads_classes)),
        target_names=tirads_classes,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_tirads = confusion_matrix(
    y_tirads_true,
    y_tirads_pred,
    labels=np.arange(len(tirads_classes))
)

print("\nConfusion Matrix:")
print(cm_tirads)

Best TIRADS classifier loaded successfully.
Best epoch: 3
Best validation loss: 1.6851

TIRADS Test Evaluation
Accuracy : 0.1081
Precision: 0.0197
Recall   : 0.1081
F1-Score : 0.0332

Classification Report:
              precision    recall  f1-score   support

           2       0.12      0.75      0.21         4
           3       0.00      0.00      0.00         6
          4a       0.00      0.00      0.00        14
          4b       0.00      0.00      0.00         7
          4c       0.08      0.33      0.13         3
           5       0.00      0.00      0.00         3

    accuracy                           0.11        37
   macro avg       0.03      0.18      0.06        37
weighted avg       0.02      0.11      0.03        37


Confusion Matrix:
[[3 0 0 0 1 0]
 [6 0 0 0 0 0]
 [7 0 0 0 7 0]
 [4 0 0 0 3 0]
 [2 0 0 0 1 0]
 [3 0 0 0 0 0]]


In [20]:
# ============================================================
# Step 11 - Improved TIRADS Training Setup
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
import numpy as np

# ------------------------------------------------------------
# Class counts
# ------------------------------------------------------------

train_counts = (
    tirads_train_df["tirads"]
    .value_counts()
    .reindex(tirads_classes)
    .fillna(0)
)

print("Training class counts:")
print(train_counts)


# ------------------------------------------------------------
# Sample weights for balanced sampling
# ------------------------------------------------------------

class_weight_map = {
    class_name: 1.0 / count
    for class_name, count in train_counts.items()
    if count > 0
}

sample_weights = np.array([
    class_weight_map[
        str(label)
    ]
    for label in tirads_train_df["tirads"]
], dtype=np.float64)

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ------------------------------------------------------------
# Rebuild training DataLoader
# ------------------------------------------------------------

tirads_train_loader = DataLoader(
    tirads_train_dataset,
    batch_size=8,
    sampler=balanced_sampler,
    num_workers=0
)

# ------------------------------------------------------------
# Reinitialize classifier
# ------------------------------------------------------------

tirads_model = TIRADSClassifier(
    num_classes=NUM_TIRADS_CLASSES
).to(device)

# ------------------------------------------------------------
# Load U-Net encoder weights again
# ------------------------------------------------------------

unet_checkpoint = torch.load(
    FINAL_UNET_PATH,
    map_location=device
)

encoder_state = {}

for key, value in unet_checkpoint.items():

    if (
        key.startswith("enc1.")
        or key.startswith("enc2.")
        or key.startswith("enc3.")
        or key.startswith("bottleneck.")
    ):
        encoder_state[key] = value

tirads_model.load_state_dict(
    encoder_state,
    strict=False
)

# ------------------------------------------------------------
# Classification loss
# ------------------------------------------------------------

class_weights = (
    len(tirads_train_df)
    / (
        NUM_TIRADS_CLASSES
        * train_counts.values
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# ------------------------------------------------------------
# Lower learning rate
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    tirads_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

print("\nImproved TIRADS training setup ready.")

print("Batch size:", 8)
print("Learning rate:", 3e-4)
print("Sampler: Balanced")
print("Loss: Weighted CrossEntropy")
print("Optimizer: AdamW")
print("Device:", device)

Training class counts:
tirads
2     30
3     13
4a    73
4b    60
4c    56
5     39
Name: count, dtype: int64

Improved TIRADS training setup ready.
Batch size: 8
Learning rate: 0.0003
Sampler: Balanced
Loss: Weighted CrossEntropy
Optimizer: AdamW
Device: cpu


In [2]:
# ============================================================
# Step 12A - Restore TIRADS Notebook Environment
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image

from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# Project Root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print("Project Root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:")
print(device)


# ------------------------------------------------------------
# TIRADS classes
# ------------------------------------------------------------

tirads_classes = [
    "2",
    "3",
    "4a",
    "4b",
    "4c",
    "5"
]

tirads_to_index = {
    label: index
    for index, label in enumerate(tirads_classes)
}

index_to_tirads = {
    index: label
    for label, index in tirads_to_index.items()
}

NUM_TIRADS_CLASSES = len(tirads_classes)

print("\nTIRADS classes:")
print(tirads_classes)


# ------------------------------------------------------------
# Load clean TIRADS dataframe
# ------------------------------------------------------------

MAPPING_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "mapping.csv"
)

CLEAN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_manifest.csv"
)

mapping_df = pd.read_csv(MAPPING_PATH)
clean_df = pd.read_csv(CLEAN_MANIFEST_PATH)

tirads_df = clean_df[
    [
        "case_number",
        "image_filename",
        "processed_image_path",
        "processed_mask_path"
    ]
].merge(
    mapping_df[
        [
            "case_number",
            "image_filename",
            "tirads"
        ]
    ],
    on=[
        "case_number",
        "image_filename"
    ],
    how="left"
)

tirads_labeled_df = tirads_df[
    tirads_df["tirads"].notna()
].copy()

tirads_labeled_df["tirads"] = (
    tirads_labeled_df["tirads"]
    .astype(str)
    .str.strip()
    .str.lower()
)

tirads_labeled_df = tirads_labeled_df[
    tirads_labeled_df["tirads"].isin(
        tirads_classes
    )
].copy()

print("\nLabeled TIRADS images:")
print(len(tirads_labeled_df))


# ------------------------------------------------------------
# Re-create same case-level split
# ------------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

gss_train = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    gss_train.split(
        tirads_labeled_df,
        groups=tirads_labeled_df["case_number"]
    )
)

tirads_train_df = tirads_labeled_df.iloc[
    train_idx
].copy()

tirads_temp_df = tirads_labeled_df.iloc[
    temp_idx
].copy()

gss_val_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_val_test.split(
        tirads_temp_df,
        groups=tirads_temp_df["case_number"]
    )
)

tirads_val_df = tirads_temp_df.iloc[
    val_idx
].copy()

tirads_test_df = tirads_temp_df.iloc[
    test_idx
].copy()

print("\nDataset split:")
print("Training:", len(tirads_train_df))
print("Validation:", len(tirads_val_df))
print("Testing:", len(tirads_test_df))


# ------------------------------------------------------------
# Dataset class
# ------------------------------------------------------------

class TIRADSImageDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["processed_image_path"]
        ).convert("RGB")

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        ).permute(2, 0, 1)

        label = tirads_to_index[
            str(row["tirads"])
        ]

        label_tensor = torch.tensor(
            label,
            dtype=torch.long
        )

        return image_tensor, label_tensor


# ------------------------------------------------------------
# Datasets
# ------------------------------------------------------------

tirads_train_dataset = TIRADSImageDataset(
    tirads_train_df
)

tirads_val_dataset = TIRADSImageDataset(
    tirads_val_df
)

tirads_test_dataset = TIRADSImageDataset(
    tirads_test_df
)


# ------------------------------------------------------------
# Balanced sampler
# ------------------------------------------------------------

train_counts = (
    tirads_train_df["tirads"]
    .value_counts()
    .reindex(tirads_classes)
    .fillna(0)
)

class_weight_map = {
    label: 1.0 / count
    for label, count in train_counts.items()
    if count > 0
}

sample_weights = torch.tensor(
    [
        class_weight_map[str(label)]
        for label in tirads_train_df["tirads"]
    ],
    dtype=torch.double
)

balanced_sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

tirads_train_loader = DataLoader(
    tirads_train_dataset,
    batch_size=8,
    sampler=balanced_sampler,
    num_workers=0
)

tirads_val_loader = DataLoader(
    tirads_val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

tirads_test_loader = DataLoader(
    tirads_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# DoubleConv
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------------------------------------------------
# TIRADS Classifier
# ------------------------------------------------------------

class TIRADSClassifier(nn.Module):

    def __init__(self, num_classes=6):
        super().__init__()

        self.enc1 = DoubleConv(3, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.global_pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Dropout(0.30),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(inplace=True),

            nn.Dropout(0.20),

            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(self, x):

        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        features = self.bottleneck(p3)

        pooled = self.global_pool(
            features
        )

        output = self.classifier(
            pooled
        )

        return output


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

tirads_model = TIRADSClassifier(
    num_classes=NUM_TIRADS_CLASSES
).to(device)


# ------------------------------------------------------------
# Load final clean U-Net encoder
# ------------------------------------------------------------

FINAL_UNET_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "ultrasound_unet_clean.pth"
)

unet_checkpoint = torch.load(
    FINAL_UNET_PATH,
    map_location=device
)

encoder_state = {}

for key, value in unet_checkpoint.items():

    if (
        key.startswith("enc1.")
        or key.startswith("enc2.")
        or key.startswith("enc3.")
        or key.startswith("bottleneck.")
    ):
        encoder_state[key] = value

tirads_model.load_state_dict(
    encoder_state,
    strict=False
)


# ------------------------------------------------------------
# Weighted loss
# ------------------------------------------------------------

class_weights = (
    len(tirads_train_df)
    / (
        NUM_TIRADS_CLASSES
        * train_counts.values
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    tirads_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\nTIRADS training environment restored.")
print("Training samples:", len(tirads_train_dataset))
print("Validation samples:", len(tirads_val_dataset))
print("Testing samples:", len(tirads_test_dataset))
print("Classes:", NUM_TIRADS_CLASSES)
print("Model device:", device)

Project Root:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure

Device:
cpu

TIRADS classes:
['2', '3', '4a', '4b', '4c', '5']

Labeled TIRADS images:
339

Dataset split:
Training: 271
Validation: 31
Testing: 37

TIRADS training environment restored.
Training samples: 271
Validation samples: 31
Testing samples: 37
Classes: 6
Model device: cpu


In [3]:
# ============================================================
# Step 12B - TIRADS Classification Using U-Net Features
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# Load final clean image features
# ------------------------------------------------------------

IMAGE_FEATURE_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_image_features.csv"
)

image_features_df = pd.read_csv(
    IMAGE_FEATURE_PATH
)

print("Image features loaded:")
print("Rows:", len(image_features_df))
print("Columns:", len(image_features_df.columns))


# ------------------------------------------------------------
# Add TIRADS labels
# ------------------------------------------------------------

feature_tirads_df = image_features_df.merge(
    tirads_labeled_df[
        [
            "case_number",
            "image_filename",
            "tirads"
        ]
    ],
    on=[
        "case_number",
        "image_filename"
    ],
    how="inner"
)

print("\nTIRADS labelled feature records:")
print(len(feature_tirads_df))


# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

feature_columns = [
    column
    for column in feature_tirads_df.columns
    if column.startswith("image_feature_")
]

print("\nNumber of image features:")
print(len(feature_columns))


# ------------------------------------------------------------
# Match existing case-level splits
# ------------------------------------------------------------

train_case_set = set(
    tirads_train_df["case_number"]
)

val_case_set = set(
    tirads_val_df["case_number"]
)

test_case_set = set(
    tirads_test_df["case_number"]
)

train_features_df = feature_tirads_df[
    feature_tirads_df["case_number"].isin(train_case_set)
].copy()

val_features_df = feature_tirads_df[
    feature_tirads_df["case_number"].isin(val_case_set)
].copy()

test_features_df = feature_tirads_df[
    feature_tirads_df["case_number"].isin(test_case_set)
].copy()


# ------------------------------------------------------------
# Prepare X and y
# ------------------------------------------------------------

X_train_tirads = train_features_df[
    feature_columns
].values

X_val_tirads = val_features_df[
    feature_columns
].values

X_test_tirads = test_features_df[
    feature_columns
].values

y_train_tirads = np.array([
    tirads_to_index[str(label)]
    for label in train_features_df["tirads"]
])

y_val_tirads = np.array([
    tirads_to_index[str(label)]
    for label in val_features_df["tirads"]
])

y_test_tirads = np.array([
    tirads_to_index[str(label)]
    for label in test_features_df["tirads"]
])


# ------------------------------------------------------------
# Train Random Forest
# ------------------------------------------------------------

tirads_rf_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

print("\nTraining TIRADS Random Forest...")

tirads_rf_model.fit(
    X_train_tirads,
    y_train_tirads
)

print("TIRADS Random Forest training completed.")


# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

y_val_pred = tirads_rf_model.predict(
    X_val_tirads
)

val_accuracy = accuracy_score(
    y_val_tirads,
    y_val_pred
)

val_precision = precision_score(
    y_val_tirads,
    y_val_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

val_recall = recall_score(
    y_val_tirads,
    y_val_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)

val_f1 = f1_score(
    y_val_tirads,
    y_val_pred,
    labels=np.arange(len(tirads_classes)),
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# Validation results
# ------------------------------------------------------------

print("\nTIRADS Validation Results")
print("=" * 60)

print(f"Accuracy : {val_accuracy:.4f}")
print(f"Precision: {val_precision:.4f}")
print(f"Recall   : {val_recall:.4f}")
print(f"F1-Score : {val_f1:.4f}")


# ------------------------------------------------------------
# Validation classification report
# ------------------------------------------------------------

print("\nValidation Classification Report:")

print(
    classification_report(
        y_val_tirads,
        y_val_pred,
        labels=np.arange(len(tirads_classes)),
        target_names=tirads_classes,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Dataset information
# ------------------------------------------------------------

print("\nDataset sizes:")
print("Training:", len(train_features_df))
print("Validation:", len(val_features_df))
print("Testing:", len(test_features_df))

print("\nFeature matrix shapes:")
print("Train:", X_train_tirads.shape)
print("Validation:", X_val_tirads.shape)
print("Test:", X_test_tirads.shape)

Image features loaded:
Rows: 455
Columns: 130

TIRADS labelled feature records:
339

Number of image features:
128

Training TIRADS Random Forest...
TIRADS Random Forest training completed.

TIRADS Validation Results
Accuracy : 0.1613
Precision: 0.1146
Recall   : 0.1613
F1-Score : 0.1230

Validation Classification Report:
              precision    recall  f1-score   support

           2       0.00      0.00      0.00         7
           3       0.00      0.00      0.00         0
          4a       0.23      0.60      0.33         5
          4b       0.10      0.11      0.11         9
          4c       0.00      0.00      0.00         7
           5       0.50      0.33      0.40         3

    accuracy                           0.16        31
   macro avg       0.14      0.17      0.14        31
weighted avg       0.11      0.16      0.12        31


Dataset sizes:
Training: 271
Validation: 31
Testing: 37

Feature matrix shapes:
Train: (271, 128)
Validation: (31, 128)
Test: (37, 1

In [4]:
# ============================================================
# Step 13 - Prepare Multimodal Representations
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Load saved clinical model artifacts
# ------------------------------------------------------------

import joblib

CLINICAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_model.joblib"
)

CLINICAL_PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_preprocessor.joblib"
)

CLINICAL_ENCODER_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "target_encoder.joblib"
)

clinical_model = joblib.load(
    CLINICAL_MODEL_PATH
)

clinical_preprocessor = joblib.load(
    CLINICAL_PREPROCESSOR_PATH
)

clinical_target_encoder = joblib.load(
    CLINICAL_ENCODER_PATH
)

print("Clinical artifacts loaded.")


# ------------------------------------------------------------
# Prepare a clean clinical dataset
# ------------------------------------------------------------

clinical_model_df = clinical_df.copy()

# Remove records without diagnosis
clinical_model_df = clinical_model_df[
    clinical_model_df["target"] != "-"
].copy()

# Remove singleton target classes
clinical_class_counts = (
    clinical_model_df["target"]
    .value_counts()
)

valid_clinical_classes = (
    clinical_class_counts[
        clinical_class_counts >= 2
    ].index
)

clinical_model_df = clinical_model_df[
    clinical_model_df["target"].isin(
        valid_clinical_classes
    )
].copy()


# ------------------------------------------------------------
# Build clinical features
# ------------------------------------------------------------

clinical_X = clinical_model_df.drop(
    columns=[
        "target",
        "patient_id",
        "TBG"
    ]
).copy()

binary_columns = [
    "on_thyroxine",
    "query_on_thyroxine",
    "on_antithyroid_meds",
    "sick",
    "pregnant",
    "thyroid_surgery",
    "I131_treatment",
    "query_hypothyroid",
    "query_hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych",
    "TSH_measured",
    "T3_measured",
    "TT4_measured",
    "T4U_measured",
    "FTI_measured",
    "TBG_measured"
]

for column in binary_columns:

    clinical_X[column] = clinical_X[column].map({
        "t": 1,
        "f": 0
    })

# Sex
clinical_X["sex"] = clinical_X["sex"].map({
    "M": 1,
    "F": 0
})


# ------------------------------------------------------------
# Transform clinical data
# ------------------------------------------------------------

clinical_processed = clinical_preprocessor.transform(
    clinical_X
)

print("\nProcessed clinical shape:")
print(clinical_processed.shape)


# ------------------------------------------------------------
# Clinical prediction probabilities
# ------------------------------------------------------------

clinical_probabilities = (
    clinical_model.predict_proba(
        clinical_processed
    )
)

print("\nClinical probability matrix:")
print(clinical_probabilities.shape)


# ------------------------------------------------------------
# Ultrasound representations
# ------------------------------------------------------------

IMAGE_FEATURE_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_image_features.csv"
)

ultrasound_features_df = pd.read_csv(
    IMAGE_FEATURE_PATH
)

ultrasound_feature_columns = [
    column
    for column in ultrasound_features_df.columns
    if column.startswith("image_feature_")
]

ultrasound_features = (
    ultrasound_features_df[
        ultrasound_feature_columns
    ].values
)

print("\nUltrasound feature matrix:")
print(ultrasound_features.shape)


# ------------------------------------------------------------
# Standardize ultrasound representation
# ------------------------------------------------------------

ultrasound_scaler = StandardScaler()

ultrasound_features_scaled = (
    ultrasound_scaler.fit_transform(
        ultrasound_features
    )
)

print("\nScaled ultrasound features:")
print(ultrasound_features_scaled.shape)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nMultimodal representations prepared.")
print("=" * 60)

print(
    "Clinical representation:",
    clinical_probabilities.shape
)

print(
    "Ultrasound representation:",
    ultrasound_features_scaled.shape
)

print(
    "Clinical patients:",
    len(clinical_model_df)
)

print(
    "Ultrasound images:",
    len(ultrasound_features_df)
)

print(
    "\nImportant:"
)

print(
    "No artificial patient-to-ultrasound pairing was created."
)

Clinical artifacts loaded.


NameError: name 'clinical_df' is not defined

In [5]:
# ============================================================
# Step 13A - Reload Clinical Dataset
# ============================================================

from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Clinical dataset path
CLINICAL_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "clinical"
    / "Thyroid Disease Data"
    / "thyroidDF.csv"
)

# Load clinical dataset
if not CLINICAL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Clinical dataset not found:\n{CLINICAL_DATA_PATH}"
    )

clinical_df = pd.read_csv(
    CLINICAL_DATA_PATH
)

print("Clinical dataset loaded successfully.")
print("Rows:", len(clinical_df))
print("Columns:", len(clinical_df.columns))

Clinical dataset loaded successfully.
Rows: 9172
Columns: 31


In [6]:
# ============================================================
# Step 13 - Prepare Multimodal Representations
# ============================================================

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Load clinical model artifacts
# ------------------------------------------------------------

CLINICAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_model.joblib"
)

CLINICAL_PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_preprocessor.joblib"
)

CLINICAL_ENCODER_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "target_encoder.joblib"
)

clinical_model = joblib.load(
    CLINICAL_MODEL_PATH
)

clinical_preprocessor = joblib.load(
    CLINICAL_PREPROCESSOR_PATH
)

clinical_target_encoder = joblib.load(
    CLINICAL_ENCODER_PATH
)

print("Clinical artifacts loaded successfully.")


# ------------------------------------------------------------
# Prepare clinical records
# ------------------------------------------------------------

clinical_model_df = clinical_df[
    clinical_df["target"] != "-"
].copy()

clinical_class_counts = (
    clinical_model_df["target"]
    .value_counts()
)

valid_clinical_classes = (
    clinical_class_counts[
        clinical_class_counts >= 2
    ].index
)

clinical_model_df = clinical_model_df[
    clinical_model_df["target"].isin(
        valid_clinical_classes
    )
].copy()

print("\nClinical records for representation:")
print(len(clinical_model_df))


# ------------------------------------------------------------
# Clinical feature preparation
# ------------------------------------------------------------

clinical_X = clinical_model_df.drop(
    columns=[
        "target",
        "patient_id",
        "TBG"
    ]
).copy()

binary_columns = [
    "on_thyroxine",
    "query_on_thyroxine",
    "on_antithyroid_meds",
    "sick",
    "pregnant",
    "thyroid_surgery",
    "I131_treatment",
    "query_hypothyroid",
    "query_hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych",
    "TSH_measured",
    "T3_measured",
    "TT4_measured",
    "T4U_measured",
    "FTI_measured",
    "TBG_measured"
]

for column in binary_columns:
    clinical_X[column] = clinical_X[column].map({
        "t": 1,
        "f": 0
    })

clinical_X["sex"] = clinical_X["sex"].map({
    "M": 1,
    "F": 0
})


# ------------------------------------------------------------
# Transform using saved preprocessor
# ------------------------------------------------------------

clinical_processed = clinical_preprocessor.transform(
    clinical_X
)

print("\nProcessed clinical shape:")
print(clinical_processed.shape)


# ------------------------------------------------------------
# Clinical model probability representation
# ------------------------------------------------------------

clinical_probabilities = (
    clinical_model.predict_proba(
        clinical_processed
    )
)

print("\nClinical probability representation:")
print(clinical_probabilities.shape)


# ------------------------------------------------------------
# Load final clean ultrasound features
# ------------------------------------------------------------

IMAGE_FEATURE_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_image_features.csv"
)

ultrasound_features_df = pd.read_csv(
    IMAGE_FEATURE_PATH
)

ultrasound_feature_columns = [
    column
    for column in ultrasound_features_df.columns
    if column.startswith("image_feature_")
]

ultrasound_features = (
    ultrasound_features_df[
        ultrasound_feature_columns
    ].values
)

print("\nUltrasound feature representation:")
print(ultrasound_features.shape)


# ------------------------------------------------------------
# Scale ultrasound representation
# ------------------------------------------------------------

ultrasound_scaler = StandardScaler()

ultrasound_features_scaled = (
    ultrasound_scaler.fit_transform(
        ultrasound_features
    )
)

print("\nScaled ultrasound representation:")
print(ultrasound_features_scaled.shape)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MULTIMODAL REPRESENTATION SUMMARY")
print("=" * 60)

print(
    "Clinical representation:",
    clinical_probabilities.shape
)

print(
    "Ultrasound representation:",
    ultrasound_features_scaled.shape
)

print(
    "\nClinical patient records:",
    len(clinical_model_df)
)

print(
    "Ultrasound image records:",
    len(ultrasound_features_df)
)

print(
    "\nNo artificial patient-ultrasound pairing created."
)

Clinical artifacts loaded successfully.

Clinical records for representation:
2396

Processed clinical shape:
(2396, 34)

Clinical probability representation:
(2396, 26)

Ultrasound feature representation:
(455, 128)

Scaled ultrasound representation:
(455, 128)

MULTIMODAL REPRESENTATION SUMMARY
Clinical representation: (2396, 26)
Ultrasound representation: (455, 128)

Clinical patient records: 2396
Ultrasound image records: 455

No artificial patient-ultrasound pairing created.


In [7]:
# ============================================================
# Step 14 - Multimodal Evidence Fusion
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Clinical model output
# ------------------------------------------------------------

clinical_top_indices = np.argsort(
    clinical_probabilities,
    axis=1
)[:, ::-1]

clinical_top3 = clinical_top_indices[:, :3]

clinical_top3_probabilities = np.take_along_axis(
    clinical_probabilities,
    clinical_top3,
    axis=1
)

clinical_top3_labels = np.array(
    clinical_target_encoder.classes_
)[clinical_top3]


# ------------------------------------------------------------
# Display sample clinical predictions
# ------------------------------------------------------------

print("Clinical prediction representation created.")

for i in range(min(5, len(clinical_probabilities))):

    print("\nClinical Record", i + 1)

    for rank in range(3):

        label = clinical_top3_labels[i, rank]
        probability = (
            clinical_top3_probabilities[i, rank]
        )

        print(
            f"  Rank {rank + 1}: "
            f"{label} "
            f"({probability:.4f})"
        )


# ------------------------------------------------------------
# Ultrasound representation
# ------------------------------------------------------------

print("\nUltrasound image representation ready.")

print(
    "Ultrasound feature shape:",
    ultrasound_features_scaled.shape
)


# ------------------------------------------------------------
# Create multimodal evidence metadata
# ------------------------------------------------------------

multimodal_summary = pd.DataFrame({
    "case_number":
        ultrasound_features_df["case_number"].values,

    "image_filename":
        ultrasound_features_df["image_filename"].values
})


# ------------------------------------------------------------
# Important note
# ------------------------------------------------------------

multimodal_summary["clinical_pairing_available"] = False

multimodal_summary["fusion_method"] = (
    "Late Evidence Integration"
)

multimodal_summary["clinical_model"] = (
    "Random Forest"
)

multimodal_summary["ultrasound_model"] = (
    "UNetSmall Segmentation"
)

multimodal_summary["tirads_model_used"] = False


print("\nMultimodal evidence framework created.")

display(
    multimodal_summary.head(10)
)


# ------------------------------------------------------------
# Save fusion configuration
# ------------------------------------------------------------

FUSION_CONFIG_PATH = (
    PROJECT_ROOT
    / "models"
    / "multimodal_fusion_config.json"
)

fusion_config = {
    "fusion_type": "late_evidence_integration",
    "supervised_fusion_training": False,
    "reason": (
        "Clinical patient IDs and ultrasound case numbers "
        "have no documented common identifier."
    ),
    "clinical_model": (
        "models/clinical/clinical_model.joblib"
    ),
    "ultrasound_model": (
        "models/image/ultrasound_unet_clean.pth"
    ),
    "tirads_classifier_used": False,
    "clinical_output": (
        "Diagnosis probability distribution"
    ),
    "ultrasound_output": (
        "Segmentation-based imaging evidence"
    ),
    "final_decision_authority": (
        "Senior Doctor"
    ),
    "ai_role": (
        "Clinical decision support only"
    )
}

import json

with open(
    FUSION_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        fusion_config,
        file,
        indent=4
    )


print("\nFusion configuration saved:")
print(FUSION_CONFIG_PATH)

Clinical prediction representation created.

Clinical Record 1
  Rank 1: S (0.9867)
  Rank 2: I (0.0067)
  Rank 3: G (0.0033)

Clinical Record 2
  Rank 1: F (0.8733)
  Rank 2: M (0.0633)
  Rank 3: MK (0.0200)

Clinical Record 3
  Rank 1: AK (0.6300)
  Rank 2: R (0.2767)
  Rank 3: A (0.0700)

Clinical Record 4
  Rank 1: R (0.5533)
  Rank 2: O (0.1267)
  Rank 3: F (0.1200)

Clinical Record 5
  Rank 1: I (0.5567)
  Rank 2: L (0.1700)
  Rank 3: M (0.1400)

Ultrasound image representation ready.
Ultrasound feature shape: (455, 128)

Multimodal evidence framework created.


,case_number,image_filename,clinical_pairing_available,fusion_method,clinical_model,ultrasound_model,tirads_model_used
0,2,2_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
1,3,3_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
2,4,4_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
3,5,5_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
4,6,6_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
5,7,7_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
6,8,8_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
7,9,9_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
8,10,10_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False
9,11,11_1.jpg,False,Late Evidence Integration,Random Forest,UNetSmall Segmentation,False



Fusion configuration saved:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\multimodal_fusion_config.json


In [8]:
# ============================================================
# Step 15 - End-to-End Multimodal Inference
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

CLINICAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_model.joblib"
)

CLINICAL_PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "clinical_preprocessor.joblib"
)

CLINICAL_ENCODER_PATH = (
    PROJECT_ROOT
    / "models"
    / "clinical"
    / "target_encoder.joblib"
)

ULTRASOUND_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "image"
    / "ultrasound_unet_clean.pth"
)


# ============================================================
# 2. Load Clinical Model
# ============================================================

clinical_model = joblib.load(
    CLINICAL_MODEL_PATH
)

clinical_preprocessor = joblib.load(
    CLINICAL_PREPROCESSOR_PATH
)

clinical_target_encoder = joblib.load(
    CLINICAL_ENCODER_PATH
)

print("Clinical model loaded.")


# ============================================================
# 3. Define U-Net
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetSmall(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(3, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up3 = nn.ConvTranspose2d(
            128, 64, 2, 2
        )
        self.dec3 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(
            64, 32, 2, 2
        )
        self.dec2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(
            32, 16, 2, 2
        )
        self.dec1 = DoubleConv(32, 16)

        self.output = nn.Conv2d(
            16, out_channels, 1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        b = self.bottleneck(p3)

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# ============================================================
# 4. Load Final Ultrasound Model
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ultrasound_model = UNetSmall(
    in_channels=3,
    out_channels=1
).to(device)

ultrasound_model.load_state_dict(
    torch.load(
        ULTRASOUND_MODEL_PATH,
        map_location=device
    )
)

ultrasound_model.eval()

print("Ultrasound model loaded.")
print("Device:", device)


# ============================================================
# 5. Clinical Input Columns
# ============================================================

BINARY_COLUMNS = [
    "on_thyroxine",
    "query_on_thyroxine",
    "on_antithyroid_meds",
    "sick",
    "pregnant",
    "thyroid_surgery",
    "I131_treatment",
    "query_hypothyroid",
    "query_hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych",
    "TSH_measured",
    "T3_measured",
    "TT4_measured",
    "T4U_measured",
    "FTI_measured",
    "TBG_measured"
]

CLINICAL_COLUMNS = [
    "age",
    "sex",
    *BINARY_COLUMNS,
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI",
    "referral_source"
]


# ============================================================
# 6. Clinical Preprocessing Helper
# ============================================================

def prepare_clinical_input(clinical_input):

    row = {}

    # --------------------------------------------------------
    # Copy expected fields
    # --------------------------------------------------------

    for column in CLINICAL_COLUMNS:

        row[column] = clinical_input.get(
            column,
            np.nan
        )

    # --------------------------------------------------------
    # Binary fields
    # --------------------------------------------------------

    for column in BINARY_COLUMNS:

        value = row[column]

        if isinstance(value, bool):
            row[column] = int(value)

        elif isinstance(value, str):

            value_lower = value.strip().lower()

            if value_lower == "t":
                row[column] = 1

            elif value_lower == "f":
                row[column] = 0

            elif value_lower in {"true", "yes", "1"}:
                row[column] = 1

            elif value_lower in {"false", "no", "0"}:
                row[column] = 0

    # --------------------------------------------------------
    # Sex
    # --------------------------------------------------------

    if isinstance(row["sex"], str):

        sex_value = row["sex"].strip().upper()

        if sex_value == "M":
            row["sex"] = "M"

        elif sex_value == "F":
            row["sex"] = "F"

        else:
            row["sex"] = np.nan

    # --------------------------------------------------------
    # Convert numeric values
    # --------------------------------------------------------

    numeric_columns = [
        "age",
        "TSH",
        "T3",
        "TT4",
        "T4U",
        "FTI"
    ]

    for column in numeric_columns:

        try:

            if pd.isna(row[column]):
                continue

            row[column] = float(
                row[column]
            )

        except (TypeError, ValueError):

            row[column] = np.nan

    return pd.DataFrame(
        [row],
        columns=CLINICAL_COLUMNS
    )


# ============================================================
# 7. Multimodal Inference Function
# ============================================================

def run_multimodal_inference(
    clinical_input,
    ultrasound_image_path
):

    ultrasound_image_path = Path(
        ultrasound_image_path
    )

    if not ultrasound_image_path.exists():
        raise FileNotFoundError(
            f"Ultrasound image not found:\n"
            f"{ultrasound_image_path}"
        )

    # --------------------------------------------------------
    # Clinical inference
    # --------------------------------------------------------

    clinical_input_df = prepare_clinical_input(
        clinical_input
    )

    clinical_processed = (
        clinical_preprocessor.transform(
            clinical_input_df
        )
    )

    clinical_probabilities = (
        clinical_model.predict_proba(
            clinical_processed
        )[0]
    )

    top_indices = np.argsort(
        clinical_probabilities
    )[::-1][:3]

    clinical_predictions = []

    for index in top_indices:

        clinical_predictions.append({
            "diagnosis_code": str(
                clinical_target_encoder.classes_[index]
            ),
            "probability": float(
                clinical_probabilities[index]
            )
        })

    # --------------------------------------------------------
    # Ultrasound inference
    # --------------------------------------------------------

    image = Image.open(
        ultrasound_image_path
    ).convert("RGB")

    original_size = image.size

    image_resized = image.resize(
        (224, 224),
        Image.Resampling.BILINEAR
    )

    image_array = np.asarray(
        image_resized,
        dtype=np.float32
    ) / 255.0

    image_tensor = torch.tensor(
        image_array,
        dtype=torch.float32
    ).permute(2, 0, 1).unsqueeze(0)

    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        segmentation_logits = (
            ultrasound_model(
                image_tensor
            )
        )

        segmentation_probability = (
            torch.sigmoid(
                segmentation_logits
            )
        )

        segmentation_mask = (
            segmentation_probability >= 0.5
        ).float()

    # --------------------------------------------------------
    # Segmentation evidence
    # --------------------------------------------------------

    positive_pixels = (
        segmentation_mask.sum().item()
    )

    total_pixels = (
        segmentation_mask.numel()
    )

    lesion_area_ratio = (
        positive_pixels
        / total_pixels
    )

    # --------------------------------------------------------
    # Return evidence summary
    # --------------------------------------------------------

    result = {

        "status": "success",

        "clinical_evidence": {
            "top_predictions":
                clinical_predictions,

            "model":
                "Random Forest"
        },

        "ultrasound_evidence": {

            "model":
                "UNetSmall",

            "original_image_size":
                list(original_size),

            "processed_image_size":
                [224, 224],

            "segmentation_pixels":
                int(positive_pixels),

            "segmentation_area_ratio":
                float(
                    lesion_area_ratio
                )
        },

        "multimodal_status": {
            "clinical_data_available":
                True,

            "ultrasound_data_available":
                True,

            "direct_training_pair_available":
                False,

            "fusion_method":
                "Late Evidence Integration",

            "requires_senior_doctor_review":
                True
        }
    }

    return result


print(
    "\nEnd-to-end multimodal inference function created successfully."
)

Clinical model loaded.
Ultrasound model loaded.
Device: cpu

End-to-end multimodal inference function created successfully.


In [9]:
# ============================================================
# Step 16 - Actual Multimodal Inference Test
# ============================================================

# ------------------------------------------------------------
# Select one diagnosed clinical record
# ------------------------------------------------------------

demo_clinical_row = (
    clinical_df[
        clinical_df["target"] != "-"
    ]
    .iloc[0]
    .to_dict()
)

# ------------------------------------------------------------
# Select one clean ultrasound image
# ------------------------------------------------------------

demo_ultrasound_row = (
    clean_df
    .iloc[0]
)

demo_ultrasound_path = (
    demo_ultrasound_row["processed_image_path"]
)

print("Demo Clinical Record:")
print("Patient ID:", demo_clinical_row["patient_id"])
print("Recorded Target:", demo_clinical_row["target"])

print("\nDemo Ultrasound:")
print("Case Number:", demo_ultrasound_row["case_number"])
print("Image:", demo_ultrasound_row["image_filename"])

print("\nImportant:")
print("These two records are NOT assumed to belong to the same patient.")


# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------

demo_result = run_multimodal_inference(
    clinical_input=demo_clinical_row,
    ultrasound_image_path=demo_ultrasound_path
)


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MULTIMODAL DECISION-SUPPORT OUTPUT")
print("=" * 65)

print("\nClinical Evidence:")
print(
    json.dumps(
        demo_result["clinical_evidence"],
        indent=4
    )
)

print("\nUltrasound Evidence:")
print(
    json.dumps(
        demo_result["ultrasound_evidence"],
        indent=4
    )
)

print("\nMultimodal Status:")
print(
    json.dumps(
        demo_result["multimodal_status"],
        indent=4
    )
)

Demo Clinical Record:
Patient ID: 840803047
Recorded Target: S

Demo Ultrasound:
Case Number: 2
Image: 2_1.jpg

Important:
These two records are NOT assumed to belong to the same patient.

MULTIMODAL DECISION-SUPPORT OUTPUT

Clinical Evidence:
{
    "top_predictions": [
        {
            "diagnosis_code": "S",
            "probability": 1.0
        },
        {
            "diagnosis_code": "R",
            "probability": 0.0
        },
        {
            "diagnosis_code": "Q",
            "probability": 0.0
        }
    ],
    "model": "Random Forest"
}

Ultrasound Evidence:
{
    "model": "UNetSmall",
    "original_image_size": [
        224,
        224
    ],
    "processed_image_size": [
        224,
        224
    ],
    "segmentation_pixels": 7551,
    "segmentation_area_ratio": 0.15049027423469388
}

Multimodal Status:
{
    "clinical_data_available": true,
    "ultrasound_data_available": true,
    "direct_training_pair_available": false,
    "fusion_method": "Late Ev

In [10]:
# ============================================================
# Step 17 - Final ML Artifacts and Verification
# ============================================================

from pathlib import Path
import json
import joblib


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODELS_DIR = PROJECT_ROOT / "models"

CLINICAL_DIR = MODELS_DIR / "clinical"
IMAGE_DIR = MODELS_DIR / "image"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Final artifact paths
# ------------------------------------------------------------

final_artifacts = {
    "clinical_model":
        CLINICAL_DIR / "clinical_model.joblib",

    "clinical_preprocessor":
        CLINICAL_DIR / "clinical_preprocessor.joblib",

    "clinical_target_encoder":
        CLINICAL_DIR / "target_encoder.joblib",

    "clinical_metadata":
        CLINICAL_DIR / "feature_info.json",

    "ultrasound_model":
        IMAGE_DIR / "ultrasound_unet_clean.pth",

    "ultrasound_metadata":
        IMAGE_DIR / "ultrasound_unet_clean_info.json",

    "clean_image_features":
        PROJECT_ROOT
        / "datasets"
        / "processed"
        / "ultrasound"
        / "clean_image_features.csv",

    "clean_manifest":
        PROJECT_ROOT
        / "datasets"
        / "processed"
        / "ultrasound"
        / "clean_manifest.csv",

    "multimodal_config":
        MODELS_DIR / "multimodal_fusion_config.json"
}


# ------------------------------------------------------------
# Verify final artifacts
# ------------------------------------------------------------

print("FINAL ML ARTIFACT VERIFICATION")
print("=" * 70)

artifact_status = {}

for name, path in final_artifacts.items():

    exists = path.exists()

    artifact_status[name] = exists

    print(
        f"{name:25} : "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    print(
        f"  {path}"
    )


# ------------------------------------------------------------
# Save final ML metrics
# ------------------------------------------------------------

final_metrics = {

    "clinical_model": {
        "model": "RandomForestClassifier",
        "accuracy": float(accuracy),
        "weighted_precision": float(precision),
        "weighted_recall": float(recall),
        "weighted_f1": float(f1)
    },

    "ultrasound_segmentation": {
        "model": "UNetSmall",
        "best_epoch": int(best_epoch)
        if "best_epoch" in globals()
        else None,
        "test_dice": float(final_test_dice)
        if "final_test_dice" in globals()
        else None,
        "test_iou": float(final_test_iou)
        if "final_test_iou" in globals()
        else None
    },

    "tirads_classifier": {
        "used_in_final_fusion": False,
        "reason": (
            "Current TIRADS classifier performance "
            "was insufficient for final decision-support use."
        )
    },

    "multimodal": {
        "fusion_type":
            "Late Evidence Integration",

        "supervised_fusion_training":
            False,

        "clinical_ultrasound_common_id":
            False,

        "artificial_pairing":
            False,

        "final_decision_authority":
            "Senior Doctor",

        "ai_role":
            "Clinical Decision Support Only"
    }
}


FINAL_METRICS_PATH = (
    MODELS_DIR / "final_ml_metrics.json"
)

with open(
    FINAL_METRICS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ------------------------------------------------------------
# Save demo inference output
# ------------------------------------------------------------

DEMO_OUTPUT_PATH = (
    MODELS_DIR
    / "multimodal_demo_inference.json"
)

with open(
    DEMO_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        demo_result,
        file,
        indent=4
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL ML PIPELINE SUMMARY")
print("=" * 70)

print("\nClinical model:")
print("  Accuracy :", f"{accuracy:.4f}")
print("  Precision:", f"{precision:.4f}")
print("  Recall   :", f"{recall:.4f}")
print("  F1       :", f"{f1:.4f}")

if "final_test_dice" in globals():

    print("\nUltrasound model:")
    print(
        "  Test Dice:",
        f"{final_test_dice:.4f}"
    )
    print(
        "  Test IoU :",
        f"{final_test_iou:.4f}"
    )

print("\nTIRADS classifier:")
print("  Used in final fusion: NO")

print("\nMultimodal:")
print("  Fusion: Late Evidence Integration")
print("  Artificial patient pairing: NO")
print("  Senior Doctor review: REQUIRED")

print("\nSaved:")
print(FINAL_METRICS_PATH)
print(DEMO_OUTPUT_PATH)

print("\nAll required final artifacts present:",
      all(artifact_status.values()))

FINAL ML ARTIFACT VERIFICATION
clinical_model            : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\clinical_model.joblib
clinical_preprocessor     : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\clinical_preprocessor.joblib
clinical_target_encoder   : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\target_encoder.joblib
clinical_metadata         : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\feature_info.json
ultrasound_model          : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean.pth
ultrasound_metadata       : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean_info.json
clean_image_features      : FOUND
  D:\MultiModalCancerTherapy_Structure\MultiModalCance

NameError: name 'accuracy' is not defined

In [11]:
# ============================================================
# Step 17 - Save Final ML Metrics and Verification
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODELS_DIR = PROJECT_ROOT / "models"
CLINICAL_DIR = MODELS_DIR / "clinical"
IMAGE_DIR = MODELS_DIR / "image"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Verified final metrics
# ------------------------------------------------------------

final_metrics = {

    "clinical_model": {
        "model": "RandomForestClassifier",

        "accuracy": 0.9187,

        "weighted_precision": 0.9150,

        "weighted_recall": 0.9187,

        "weighted_f1": 0.9124
    },

    "ultrasound_segmentation": {
        "model": "UNetSmall",

        "best_epoch": 8,

        "best_validation_loss": 0.6171,

        "test_dice": 0.5490,

        "test_iou": 0.4107
    },

    "tirads_classifier": {
        "used_in_final_fusion": False,

        "reason": (
            "Current TIRADS classifier performance "
            "was insufficient for final decision-support use."
        ),

        "test_accuracy_previous": 0.1081
    },

    "multimodal": {

        "fusion_type":
            "Late Evidence Integration",

        "supervised_fusion_training":
            False,

        "clinical_ultrasound_common_id":
            False,

        "artificial_pairing":
            False,

        "clinical_records":
            2396,

        "ultrasound_images":
            455,

        "final_decision_authority":
            "Senior Doctor",

        "ai_role":
            "Clinical Decision Support Only"
    }
}


# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

FINAL_METRICS_PATH = (
    MODELS_DIR / "final_ml_metrics.json"
)

with open(
    FINAL_METRICS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ------------------------------------------------------------
# Final artifact verification
# ------------------------------------------------------------

required_artifacts = [

    CLINICAL_DIR / "clinical_model.joblib",

    CLINICAL_DIR / "clinical_preprocessor.joblib",

    CLINICAL_DIR / "target_encoder.joblib",

    CLINICAL_DIR / "feature_info.json",

    IMAGE_DIR / "ultrasound_unet_clean.pth",

    IMAGE_DIR / "ultrasound_unet_clean_info.json",

    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_image_features.csv",

    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
    / "clean_manifest.csv",

    MODELS_DIR / "multimodal_fusion_config.json",

    FINAL_METRICS_PATH
]


print("FINAL ML PIPELINE VERIFICATION")
print("=" * 70)

all_found = True

for artifact in required_artifacts:

    exists = artifact.exists()

    if not exists:
        all_found = False

    print(
        f"{'FOUND' if exists else 'MISSING':8} | "
        f"{artifact}"
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL ML RESULTS")
print("=" * 70)

print("\nClinical Model")
print("Accuracy :", 91.87, "%")
print("Precision:", 91.50, "%")
print("Recall   :", 91.87, "%")
print("F1-Score :", 91.24, "%")

print("\nUltrasound U-Net")
print("Best Epoch:", 8)
print("Test Dice :", 0.5490)
print("Test IoU  :", 0.4107)

print("\nTIRADS Classifier")
print("Included in final fusion: NO")

print("\nMultimodal")
print("Fusion method: Late Evidence Integration")
print("Artificial patient pairing: NO")
print("Senior Doctor review: REQUIRED")

print("\nFinal metrics saved:")
print(FINAL_METRICS_PATH)

print("\nAll required ML artifacts present:")
print(all_found)

FINAL ML PIPELINE VERIFICATION
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\clinical_model.joblib
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\clinical_preprocessor.joblib
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\target_encoder.joblib
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\clinical\feature_info.json
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean.pth
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean_info.json
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\ultrasound\clean_image_features.csv
FOUND    | D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\processed\